# **Preparing Dependancies**

In [ ]:
# Install dependencies (jalankan sekali pada Colab/Kaggle GPU runtime)
!pip install -q diffusers==0.27.2 transformers==4.41.2 accelerate==0.30.1 safetensors
!pip install -q pillow matplotlib numpy

import torch, gc, os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFilter
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionInpaintPipeline,
    StableDiffusionImg2ImgPipeline,
    EulerAncestralDiscreteScheduler,
    DPMSolverMultistepScheduler,
    DDIMScheduler,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"Device: {DEVICE} | dtype: {DTYPE}")

# Prompt & seed yang akan dipakai konsisten sepanjang notebook
# Target: astronot di permukaan bulan dengan bumi di latar belakang (gaya ilustrasi)
PROMPT = (
    "an astronaut in a white spacesuit standing on the moon surface, "
    "planet earth visible in the sky on the upper right, starry black background, "
    "cartoon illustration, flat colors, clean digital art, vector style"
)
NEG_PROMPT = (
    "photorealistic, realistic, photograph, 3d render, messy, blurry, "
    "low quality, bad art, ugly, sketch, grainy, unfinished, chromatic aberration"
)
SEED = 222


# **Kriteria 1: Melakukan Image Generation dari Teks (Text-to-Image)**

## **Load Base Pipeline Model**

In [ ]:
# Memuat Stable Diffusion v1.5 sebagai pipeline dasar
MODEL_ID = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    safety_checker=None,
    requires_safety_checker=False,
).to(DEVICE)

pipe.set_progress_bar_config(disable=False)
print("Base pipeline siap.")


## **Generate Image**

In [ ]:
# === generate_simple_image() ===
# Parameter dasar: prompt, negative_prompt, seed
def generate_simple_image(prompt, negative_prompt, seed):
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
    ).images[0]
    return image


img_simple = generate_simple_image(PROMPT, NEG_PROMPT, SEED)
img_simple


## **Generate Image with Hyperparameter Configuration**

In [ ]:
# === generate_advanced_image() ===
# Parameter tambahan: guidance_scale, num_inference_steps
def generate_advanced_image(
    prompt,
    negative_prompt,
    seed,
    guidance_scale=8.5,
    num_inference_steps=40,
):
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    ).images[0]
    return image


img_advanced = generate_advanced_image(
    PROMPT, NEG_PROMPT, SEED,
    guidance_scale=8.5,
    num_inference_steps=40,
)
img_advanced


## **Guidance Scale Comparison**

In [ ]:
# Helper untuk menampilkan beberapa gambar sebagai grid
def show_grid(images, titles, cols=None, figsize=None):
    n = len(images)
    cols = cols or n
    rows = (n + cols - 1) // cols
    figsize = figsize or (4 * cols, 4 * rows)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax, img, t in zip(axes, images, titles):
        ax.imshow(img); ax.set_title(t); ax.axis("off")
    for ax in axes[len(images):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()


# Eksperimen guidance_scale: rendah, sedang, tinggi
cfg_values = [2.0, 7.5, 15.0]
imgs_cfg = [
    generate_advanced_image(PROMPT, NEG_PROMPT, SEED, guidance_scale=c, num_inference_steps=30)
    for c in cfg_values
]
show_grid(imgs_cfg, [f"CFG = {c}" for c in cfg_values])


### **Guidance Scale Explanation:**

*   **Gambar dengan "Scale" Rendah (CFG = 2.0):**
    Hasil cenderung lebih bebas dan kreatif, namun korespondensinya dengan prompt sangat lemah. Beberapa elemen kunci pada prompt (misal "wizard hat" atau "magical glowing forest") sering tidak muncul atau muncul tidak jelas. Komposisi terasa lebih "natural" dan variatif tetapi sering kehilangan fokus subjek utama, dan detail terlihat kurang tajam karena model kurang dipaksa mengikuti teks.

*   **Gambar dengan "Scale" Tinggi (CFG = 15.0):**
    Sebaliknya, model sangat patuh pada prompt: hampir semua elemen yang diminta muncul (rubah, topi penyihir, hutan magis, warna terang). Detail cenderung lebih tegas dan kontras lebih kuat. Namun konsekuensinya adalah munculnya artefak over-saturated, garis kontur yang terlalu tajam ("burned" look), dan tekstur yang kadang terasa tidak natural. Variasi visual berkurang karena ruang sampling-nya menjadi lebih sempit.

Sweet spot untuk Stable Diffusion 1.5 biasanya berada di sekitar CFG 7-10: cukup mengikuti prompt tanpa kehilangan kualitas estetik.


## **Inference Steps Comparison**

In [ ]:
# Eksperimen jumlah inference steps: rendah vs tinggi
step_values = [10, 30, 50]
imgs_steps = [
    generate_advanced_image(PROMPT, NEG_PROMPT, SEED, guidance_scale=7.5, num_inference_steps=s)
    for s in step_values
]
show_grid(imgs_steps, [f"Steps = {s}" for s in step_values])


### **Inference Step Explanation:**

*   **Gambar dengan "Step" Rendah (10 steps):**
    Komposisi keseluruhan sudah terbentuk, tetapi banyak area yang masih "setengah jadi": tekstur bulu rubah terlihat blocky, latar hutan tampak buram, dan terkadang muncul artefak noise/grain pada area gelap. Detail halus seperti mata, ujung topi, atau cahaya partikel sering belum terbentuk dengan baik karena denoising dihentikan terlalu dini.

*   **Gambar dengan "Step" Tinggi (50 steps):**
    Hasil jauh lebih halus dan stabil. Tekstur, gradasi pencahayaan, serta detail mikro (mata, urat daun, partikel cahaya) menjadi lebih kaya. Perbedaan dari step 30 ke 50 biasanya kecil dan mengikuti hukum *diminishing returns*: setelah 30-40 step, peningkatan visual sangat marjinal sementara waktu komputasi tumbuh linear. Untuk Stable Diffusion 1.5, 25-40 step umumnya sudah optimal.


## **Batch Inference from One Prompt**

In [ ]:
# Batch inference: 4 gambar sekaligus dari 1 prompt
def generate_batch(prompt, negative_prompt, seed, num_images=4,
                   guidance_scale=7.5, num_inference_steps=30):
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    out = pipe(
        prompt=[prompt] * num_images,
        negative_prompt=[negative_prompt] * num_images,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    )
    return out.images


batch_imgs = generate_batch(PROMPT, NEG_PROMPT, SEED, num_images=4)
show_grid(batch_imgs, [f"Img {i+1}" for i in range(4)], cols=2, figsize=(10, 10))


## **Load Scheduler**

In [ ]:
# === load_scheduler() — pergantian sampler tanpa reload model ===
def load_scheduler(pipe, scheduler_name):
    name = scheduler_name.lower().replace(" ", "").replace("-", "")
    cfg = pipe.scheduler.config
    if name in ("eulera", "eulerancestral", "euler_a"):
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(cfg)
    elif name in ("dpm++", "dpmpp", "dpmsolver++", "dpmsolverpp"):
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(
            cfg, algorithm_type="dpmsolver++"
        )
    elif name == "ddim":
        pipe.scheduler = DDIMScheduler.from_config(cfg)
    else:
        raise ValueError(f"Scheduler tidak dikenal: {scheduler_name}")
    return pipe


# Bandingkan tiga scheduler
schedulers = ["Euler A", "DPM++", "DDIM"]
imgs_sched = []
for name in schedulers:
    pipe = load_scheduler(pipe, name)
    imgs_sched.append(
        generate_advanced_image(PROMPT, NEG_PROMPT, SEED,
                                guidance_scale=7.5, num_inference_steps=30)
    )

show_grid(imgs_sched, schedulers)


### **Scheduler Comparation:**

*   **Gambar dengan "Euler A Scheduler":**
    Euler Ancestral menambahkan komponen stochastic pada setiap step sehingga hasilnya cenderung lebih artistik, ekspresif, dan "fresh". Tekstur terasa hidup dan komposisi sedikit berbeda dari sampler deterministic. Cocok untuk gaya ilustrasi/concept art seperti prompt rubah-penyihir di atas. Konvergen cepat, hasil layak sudah terlihat di ~20 step.

*   **Gambar dengan "DPM++ Scheduler":**
    DPM-Solver++ adalah multi-step ODE solver yang sangat efisien. Hasilnya tajam, detail bagus, dan sangat konsisten antar-step. Bisa menghasilkan gambar berkualitas tinggi hanya dengan 15-25 step. Karakter visual cenderung sedikit lebih "clean" dan kurang "painterly" dibanding Euler A. Pilihan default yang aman untuk produksi.

*   **Gambar dengan "DDIM Scheduler":**
    DDIM adalah sampler deterministic klasik. Hasilnya stabil dan reproducible (dengan seed sama selalu identik). Membutuhkan jumlah step lebih tinggi (30-50) untuk mencapai kualitas setara DPM++ atau Euler A. Detail bagus tetapi terkadang terlihat sedikit lebih "soft" pada area tekstur kompleks. Sangat berguna untuk eksperimen reproducible (mis. perbandingan parameter).


# **Kriteria 2: Menyempurnakan Gambar Melalui Image-to-Image**

## **Base + Refiner Image Generation**

In [ ]:
# === Refiner Pattern: Two-Stage Generation (Base + Img2Img refiner) ===
# Stage 1 menghasilkan latent menggunakan pipeline Base hingga ~80% denoising,
# Stage 2 mendekode dan menyempurnakan via Img2Img dengan strength rendah (0.2)
# yang setara dengan denoising_start = 0.8 pada paradigma SDXL.

# Img2Img pipeline reuse komponen pipe Base — tidak perlu load model baru
img2img_pipe = StableDiffusionImg2ImgPipeline(**pipe.components).to(DEVICE)
img2img_pipe.set_progress_bar_config(disable=False)


def two_stage_generation(
    prompt, negative_prompt, seed,
    total_steps=40, denoising_split=0.8,
    guidance_scale=7.5,
):
    base_steps = max(1, int(total_steps * denoising_split))

    # ---- Stage 1: Base pipeline -> latent (denoising_end = 0.8 setara) ----
    gen1 = torch.Generator(device=DEVICE).manual_seed(seed)
    latent = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=gen1,
        num_inference_steps=base_steps,
        guidance_scale=guidance_scale,
        output_type="latent",
    ).images

    # Decode latent ke PIL agar bisa dipakai Img2Img
    with torch.no_grad():
        decoded = pipe.vae.decode(latent / pipe.vae.config.scaling_factor).sample
    base_image = pipe.image_processor.postprocess(decoded, output_type="pil")[0]

    # ---- Stage 2: Img2Img refiner (denoising_start = 0.8 setara) ----
    gen2 = torch.Generator(device=DEVICE).manual_seed(seed)
    refined = img2img_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=base_image,
        strength=1.0 - denoising_split,   # 0.2
        num_inference_steps=total_steps,
        guidance_scale=guidance_scale,
        generator=gen2,
    ).images[0]

    return base_image, refined


base_only, refined = two_stage_generation(PROMPT, NEG_PROMPT, SEED)
show_grid([base_only, refined], ["Stage 1 (Base, 80% denoise)", "Stage 2 (+ Refiner)"])


## **Inpainting**

### **Load Model Inpainting**

In [ ]:
# Memuat model khusus untuk inpainting
INPAINT_MODEL_ID = "runwayml/stable-diffusion-inpainting"

inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    INPAINT_MODEL_ID,
    torch_dtype=DTYPE,
    safety_checker=None,
    requires_safety_checker=False,
).to(DEVICE)
inpaint_pipe.set_progress_bar_config(disable=False)


def inpaint_engine(image, mask, prompt,
                   negative_prompt=NEG_PROMPT,
                   seed=9,
                   guidance_scale=8.0,
                   num_inference_steps=40):
    """Fungsi inti inpainting — menerima image, mask, prompt."""
    if image.mode != "RGB":
        image = image.convert("RGB")
    if mask.mode != "L":
        mask = mask.convert("L")
    if image.size != mask.size:
        mask = mask.resize(image.size, resample=Image.NEAREST)

    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    out = inpaint_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=image,
        mask_image=mask,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    ).images[0]
    return out

print("Inpaint pipeline siap.")


### **Manual Masking**

In [ ]:
# === Manual masking (hardcoded box, trial & error) ===
# Catatan: gambar dasar untuk inpainting adalah file yang disediakan template
# submission. Letakkan file gambar tersebut di working directory dan sesuaikan
# nama file di bawah. Sebagai fallback, kita generate satu gambar bumi-dari-luar
# angkasa agar notebook tetap runnable.
BASE_IMG_PATH = "base_inpaint.jpg"

if os.path.exists(BASE_IMG_PATH):
    base_image = Image.open(BASE_IMG_PATH).convert("RGB").resize((512, 512))
else:
    print("base_inpaint.jpg tidak ditemukan -> generate gambar dasar dummy.")
    base_image = generate_advanced_image(
        prompt="planet earth viewed from outer space, stars in the background, "
               "highly detailed, cinematic lighting, digital art",
        negative_prompt=NEG_PROMPT,
        seed=9,
        guidance_scale=8.0,
        num_inference_steps=40,
    )

W, H = base_image.size
print("Ukuran gambar dasar:", base_image.size)

# Mask hardcode: kotak di sisi kanan-atas tempat satelit akan diletakkan.
# Nilai (left, top, right, bottom) ditentukan trial & error.
manual_mask = Image.new("L", (W, H), 0)
draw = ImageDraw.Draw(manual_mask)
box = (int(W * 0.55), int(H * 0.18), int(W * 0.92), int(H * 0.50))
draw.rectangle(box, fill=255)

# Visual cek
preview = base_image.copy()
overlay = Image.new("RGBA", (W, H), (255, 0, 0, 0))
ImageDraw.Draw(overlay).rectangle(box, fill=(255, 0, 0, 90))
preview = Image.alpha_composite(preview.convert("RGBA"), overlay).convert("RGB")
show_grid([base_image, manual_mask, preview],
          ["Base Image", "Manual Mask", "Mask Overlay"], cols=3)


### **Generate**

In [ ]:
# === Generate inpainting hasil manual mask ===
INPAINT_PROMPT = (
    "a broken satellite floating in space, damaged solar panels, "
    "metal debris, sparks, wires hanging, sci-fi, highly detailed, digital art"
)

result_manual = inpaint_engine(
    image=base_image,
    mask=manual_mask,
    prompt=INPAINT_PROMPT,
    negative_prompt=NEG_PROMPT,
    seed=9,
    guidance_scale=8.0,
    num_inference_steps=40,
)

show_grid([base_image, manual_mask, result_manual],
          ["Original", "Manual Mask", "Inpainted (broken satellite)"], cols=3)


## **Inpainting Menggunakan Automasking**

### **load Model Segmentation Untuk Masking**

In [ ]:
# === Load model segmentation untuk auto-masking ===
# CLIPSeg menerima prompt teks dan menghasilkan mask area yang sesuai deskripsi.
!pip install -q transformers --upgrade
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation

SEG_MODEL_ID = "CIDAS/clipseg-rd64-refined"
seg_processor = CLIPSegProcessor.from_pretrained(SEG_MODEL_ID)
seg_model = CLIPSegForImageSegmentation.from_pretrained(SEG_MODEL_ID).to(DEVICE)
seg_model.eval()
print("Segmentation model siap.")


### **Masking with Segmentation Model**

In [ ]:
# === Auto-masking dengan CLIPSeg ===
def auto_mask(image, target_prompt, threshold=0.4, dilate=10):
    inputs = seg_processor(
        text=[target_prompt],
        images=[image],
        padding=True,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        logits = seg_model(**inputs).logits  # shape (H, W) atau (1, H, W)
    if logits.ndim == 3:
        logits = logits[0]

    prob = torch.sigmoid(logits).cpu().numpy()
    binary = (prob > threshold).astype("uint8") * 255

    mask_img = Image.fromarray(binary, mode="L").resize(image.size, Image.NEAREST)
    if dilate > 0:
        mask_img = mask_img.filter(ImageFilter.MaxFilter(dilate * 2 + 1))
    return mask_img


# Contoh: target objek "earth" pada gambar dasar -> kita ingin replace bumi
auto_mask_img = auto_mask(base_image, "the planet", threshold=0.4, dilate=8)
show_grid([base_image, auto_mask_img],
          ["Base Image", "Auto Mask (CLIPSeg)"], cols=2)


### **Generate**

In [ ]:
# === Generate inpainting menggunakan auto mask ===
result_auto = inpaint_engine(
    image=base_image,
    mask=auto_mask_img,
    prompt=INPAINT_PROMPT,
    negative_prompt=NEG_PROMPT,
    seed=9,
    guidance_scale=8.0,
    num_inference_steps=40,
)

show_grid([base_image, auto_mask_img, result_auto],
          ["Original", "Auto Mask", "Inpainted (auto)"], cols=3)


## **Outpainting**

### **Prepare the Canvas**

In [ ]:
# === prepare_outpainting() — perluas kanvas ke satu arah ===
def prepare_outpainting(image, direction="right", expand_pixels=256, blur_radius=40):
    """Memperluas kanvas ke arah `direction` (left/right/top/bottom).
    Mengembalikan (canvas, mask) yang siap dikirim ke inpaint_engine."""
    if image.mode != "RGB":
        image = image.convert("RGB")
    w, h = image.size
    if direction in ("left", "right"):
        new_w, new_h = w + expand_pixels, h
    elif direction in ("top", "bottom"):
        new_w, new_h = w, h + expand_pixels
    else:
        raise ValueError(f"direction harus left/right/top/bottom, dapat {direction}")

    # Resolusi harus kelipatan 8 untuk SD
    new_w -= new_w % 8
    new_h -= new_h % 8

    # Background blur sebagai konteks warna (membantu model menebak palet)
    bg = image.resize((new_w, new_h), Image.BICUBIC).filter(
        ImageFilter.GaussianBlur(radius=blur_radius)
    )
    canvas = bg.copy()

    if direction == "right":
        paste_xy = (0, 0)
    elif direction == "left":
        paste_xy = (new_w - w, 0)
    elif direction == "top":
        paste_xy = (0, new_h - h)
    else:  # bottom
        paste_xy = (0, 0)

    canvas.paste(image, paste_xy)

    # Mask: putih = area yang akan di-generate (kosong), hitam = pertahankan
    mask = Image.new("L", (new_w, new_h), 255)
    mask.paste(Image.new("L", (w, h), 0), paste_xy)

    return canvas, mask


# Demo: perluas ke kanan 256px
out_canvas, out_mask = prepare_outpainting(result_manual, direction="right", expand_pixels=256)
show_grid([result_manual, out_canvas, out_mask],
          ["Source", "Canvas (right +256px)", "Mask"], cols=3)


### **Generate**

In [ ]:
# === Generate outpainting satu sisi ===
OUTPAINT_PROMPT = (
    "wide view of outer space, broken satellite, planet earth in background, "
    "stars, nebula, cosmic dust, highly detailed, sci-fi digital art"
)

outpainted = inpaint_engine(
    image=out_canvas,
    mask=out_mask,
    prompt=OUTPAINT_PROMPT,
    negative_prompt=NEG_PROMPT,
    seed=9,
    guidance_scale=8.0,
    num_inference_steps=40,
)

show_grid([result_manual, outpainted],
          ["Sebelum Outpaint", "Setelah Outpaint (kanan)"], cols=2)


## **Outpainting Zoom Out**

### **Prepare Canvas for Zoom Out**

In [ ]:
# === prepare_zoom_out() — perluas kanvas ke seluruh sisi sekaligus ===
def prepare_zoom_out(image, expand_pixels=128, blur_radius=50):
    """Versi 'Zoom Out': menambahkan padding di keempat sisi."""
    if image.mode != "RGB":
        image = image.convert("RGB")
    w, h = image.size
    new_w = w + 2 * expand_pixels
    new_h = h + 2 * expand_pixels
    new_w -= new_w % 8
    new_h -= new_h % 8

    bg = image.resize((new_w, new_h), Image.BICUBIC).filter(
        ImageFilter.GaussianBlur(radius=blur_radius)
    )
    canvas = bg.copy()
    paste_x = (new_w - w) // 2
    paste_y = (new_h - h) // 2
    canvas.paste(image, (paste_x, paste_y))

    mask = Image.new("L", (new_w, new_h), 255)
    mask.paste(Image.new("L", (w, h), 0), (paste_x, paste_y))
    return canvas, mask


zoom_canvas, zoom_mask = prepare_zoom_out(result_manual, expand_pixels=128)
show_grid([result_manual, zoom_canvas, zoom_mask],
          ["Source", "Zoom-Out Canvas", "Mask"], cols=3)


### **Generate**

In [ ]:
# === Generate Zoom Out (multi-step expansion) ===
# Gambar diperluas bertahap: setiap iterasi menambah 128px di seluruh sisi.
current = result_manual
zoom_outputs = [current]

for step in range(2):  # 2x zoom-out -> diperluas total 256px tiap sisi
    canvas, mask = prepare_zoom_out(current, expand_pixels=128)
    current = inpaint_engine(
        image=canvas,
        mask=mask,
        prompt=OUTPAINT_PROMPT,
        negative_prompt=NEG_PROMPT,
        seed=9 + step,
        guidance_scale=8.0,
        num_inference_steps=40,
    )
    zoom_outputs.append(current)

show_grid(
    zoom_outputs,
    [f"Step {i}" for i in range(len(zoom_outputs))],
    cols=len(zoom_outputs),
    figsize=(5 * len(zoom_outputs), 5),
)
